# Ý tưởng: tải model ViT pretrain trên tập ImageNet. Đóng băng trọng số các lớp, mở khóa lớp cuối cùng để train trên tập ISIC2018

# Pip install

In [9]:
#thư viện làm ViT
!pip install timm

#thư viện đọc file csv
!pip install pandas

#xem kiến trúc model
!pip install torchinfo

#Thư viện torch GPU
#!pip uninstall torch torchvision torchaudio -y
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

  Using cached torchinfo-1.8.0-py3-none-any.whl.metadata (21 kB)
Using cached torchinfo-1.8.0-py3-none-any.whl (23 kB)
Looking in indexes: https://download.pytorch.org/whl/cu118


# Mô hình ViT

In [2]:
import torch
import timm

# 1. Kiểm tra device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 2. Tải mô hình ViT pretrained trên ImageNet
# vit_base_patch16_224 là model phổ biến nhất
model = timm.create_model(
    'vit_base_patch16_224',
    pretrained=True,      # dùng weight pretrained
    num_classes=1000      # ImageNet (sẽ đổi khi train HAM10000)
)

# 3. Đưa model lên device
model = model.to(device)

# 4. In cấu trúc mô hình
from torchinfo import summary

summary(
    model,
    input_size=(1, 3, 224, 224),
    col_names=["input_size", "output_size", "num_params"]
)


c:\Users\hieut\Desktop\thuctap_Neuraltrans\ISIC\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


Layer (type:depth-idx)                   Input Shape               Output Shape              Param #
VisionTransformer                        [1, 3, 224, 224]          [1, 1000]                 152,064
├─PatchEmbed: 1-1                        [1, 3, 224, 224]          [1, 196, 768]             --
│    └─Conv2d: 2-1                       [1, 3, 224, 224]          [1, 768, 14, 14]          590,592
│    └─Identity: 2-2                     [1, 196, 768]             [1, 196, 768]             --
├─Dropout: 1-2                           [1, 197, 768]             [1, 197, 768]             --
├─Identity: 1-3                          [1, 197, 768]             [1, 197, 768]             --
├─Identity: 1-4                          [1, 197, 768]             [1, 197, 768]             --
├─Sequential: 1-5                        [1, 197, 768]             [1, 197, 768]             --
│    └─Block: 2-3                        [1, 197, 768]             [1, 197, 768]             --
│    │    └─LayerNorm: 3-

In [3]:
print(model)

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False

# Load dataset ISIC 2018 – Task 3

In [3]:
import os
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms




class ISIC2018Task3Dataset(Dataset):
    def __init__(self, img_dir, csv_file, transform=None):
        """
        img_dir  : thư mục ảnh
        csv_file : file ground truth CSV
        transform: torchvision transforms
        """
        self.img_dir = img_dir
        self.transform = transform

        self.df = pd.read_csv(csv_file)

        # Tên ảnh (không có .jpg)
        self.image_ids = self.df["image"].values

        # 7 lớp (one-hot → label index)
        self.class_names = self.df.columns[1:].tolist()
        self.labels = self.df.iloc[:, 1:].values.argmax(axis=1)

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_name = self.image_ids[idx] + ".jpg"
        img_path = os.path.join(self.img_dir, img_name)

        image = Image.open(img_path).convert("RGB")
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label


## Transform cho ViT (chuẩn ImageNet)

In [4]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    #transforms.RandomHorizontalFlip(),
    #transforms.RandomVerticalFlip(),
    #transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    )
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    )
])


## Load TRAIN / VAL / TEST datasets

In [5]:


train_dataset = ISIC2018Task3Dataset(
    img_dir=os.path.join("dataset", "ISIC2018_Task3_Training_Input"),
    csv_file=os.path.join("dataset/ISIC2018_Task3_Training_GroundTruth", "ISIC2018_Task3_Training_GroundTruth.csv"),
    transform=train_transform
)

val_dataset = ISIC2018Task3Dataset(
    img_dir=os.path.join("dataset", "ISIC2018_Task3_Validation_Input"),
    csv_file=os.path.join("dataset/ISIC2018_Task3_Validation_GroundTruth", "ISIC2018_Task3_Validation_GroundTruth.csv"),
    transform=val_test_transform
)

test_dataset = ISIC2018Task3Dataset(
    img_dir=os.path.join("dataset", "ISIC2018_Task3_Test_Input"),
    csv_file=os.path.join("dataset/ISIC2018_Task3_Test_GroundTruth", "ISIC2018_Task3_Test_GroundTruth.csv"),
    transform=val_test_transform
)


In [6]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)


In [7]:
print("Train samples:", len(train_dataset))
print("Val samples  :", len(val_dataset))
print("Test samples :", len(test_dataset))
print("Classes:", train_dataset.class_names)

images, labels = next(iter(train_loader))
print("Batch image shape:", images.shape)   # [B, 3, 224, 224]
print("Batch label shape:", labels.shape)   # [B]


Train samples: 10015
Val samples  : 193
Test samples : 1512
Classes: ['MEL', 'NV', 'BCC', 'AKIEC', 'BKL', 'DF', 'VASC']
Batch image shape: torch.Size([16, 3, 224, 224])
Batch label shape: torch.Size([16])


In [8]:
class_map = {
    0: "MEL",
    1: "NV",
    2: "BCC",
    3: "AKIEC",
    4: "BKL",
    5: "DF",
    6: "VASC"
}
